<a href="https://colab.research.google.com/github/BraidoLuis/estudos-yolo/blob/main/Blur%2C_mapa_de_calor_e_dist%C3%A2ncia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00


APLICANDO BLUR

---



In [ ]:
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from ultralytics.utils.plotting import Annotator, colors

model = YOLO('yolov8n.pt')

names = model.names

cap = cv2.VideoCapture("file.mp4")
assert cap.isOpened()

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

blur_ratio = 50

video_writter = cv2.VideoWriter("object_blurring_ooutput.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

while cap.isOpened():
  success, im0 = cap.read()
  if not success:
    print("Video frame is empty or video processing has been sucessfully completed.")
    break

  results = model.predict(im0, show=False)
  boxes = results[0].boxes.xyxy.cpu().tolist()
  clss = results[0].boxes.cls.cpu().tolist()
  annotator = Annotator(im0, line_width=2, example=names)

  if boxes is not None:
    for box, cls in zip(boxes, clss):
      annotator.box_label(box, color=colors(cls, True), label=names[int(cls)])

      if names[int(cls)] == "person":
        obj = im0[int(box[1]) : int(box[3]), int(box[0]) : int(box[2])]

        blur_obj = cv2.blur(obj, (blur_ratio, blur_ratio))

        im0[int(box[1]) : int(box[3]), int(box[0]) : int(box[2])] = blur_obj

    cv2_imshow(im0)

    video_writter.write(im0)

cap.release()
video_writter.release()
cv2.destroyAllWindows()

Aplicando mapa de calor

---



In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from ultralytics import solutions

cap = cv2.VideoCapture("file.mp4")
assert cap.isOpened(), "Error reading video file"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

video_writer = cv2.VideoWriter(
    "heatmap_output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

heatmap = solutions.Heatmap(
    model="yolov8n.pt",
    colormap=cv2.COLORMAP_PARULA,
    show=False
)

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Finalizado.")
        break

    result = heatmap(im0)

    if hasattr(result, "plot_im"):
        frame = result.plot_im
    else:
        frame = result

    video_writer.write(frame)
    cv2_imshow(frame)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Determinando distância

---



In [ ]:
import math
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from ultralytics.utils.plotting import Annotator

model = YOLO('yolov8n.pt')

cap = cv2.VideoCapture("/content/video3.mp4")
assert cap.isOpened(), "Error"

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

out = cv2.VideoWriter("distance_calculation.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

center_point = (0, h)
pixel_per_meter = 10

txt_color, txt_background, bbox_clr = ((0, 0, 0), (255, 255, 255), (255, 0 ,255))

while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Finalizado.")
        break

    annotator = Annotator(im0, line_width=2)

    results = model.track(im0, persist=True)
    boxes = results[0].boxes.xyxy.cpu()

    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            annotator.box_label(box, label=str(track_id), color=bbox_clr)

            x1 = int((box[0] + box[2]) // 2)
            y1 = int((box[1] + box[3]) // 2)

            distance = math.sqrt(
                (x1 - center_point[0])**2 +
                (y1 - center_point[1])**2
            ) / pixel_per_meter

            text = f"{distance:.2f} m"
            text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)

            cv2.rectangle(
                im0,
                (x1, y1 - text_size[0][1] - 10),
                (x1 + text_size[0][0] + 10, y1),
                txt_background,
                -1
            )

            cv2.putText(
                im0,
                text,
                (x1, y1 - 5),
                cv2.FONT_HERSHEY_COMPLEX,
                1.2,
                txt_color,
                3
            )

    out.write(im0)
    cv2_imshow(im0)

    if cv2.waitKey(1) & 0xFF == ord("q"):
      break

cap.release()
out.release()
cv2.destroyAllWindows()